<a href="https://colab.research.google.com/github/aasrithavalluri30-source/Adversarial-Robustness-Evaluation-of-ML-based-Network-Intrusion-Detection-/blob/main/GENAI_final_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GAN + Diffusion Adversarial Evolution vs. an IDS Classifier

**Goal:** show that a static, single-shot ML intrusion detector degrades as attack
traffic is adversarially evolved (GAN-generated, then diffusion-mutated across
generations) — motivating the need for more sophisticated / adaptive detection
techniques.

**Dataset:** UNSW-NB15 (network flow features, binary + multi-class attack labels).



In [ ]:
!pip install kagglehub --quiet

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load training set
df_train = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "mrwellsdavid/unsw-nb15",
    "UNSW_NB15_training-set.csv"
)

# Load testing set
df_test = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "mrwellsdavid/unsw-nb15",
    "UNSW_NB15_testing-set.csv"
)

print("Train:", df_train.shape)
print("Test: ", df_test.shape)
print("\nColumns:", df_train.columns.tolist())
print("\nAttack types:\n", df_train["attack_cat"].value_counts())


## 1. Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

print("Shape:", df_train.shape)
print("\nMissing values:", df_train.isnull().sum().sum())
print("\nDtypes:\n", df_train.dtypes.value_counts())


In [ ]:
# Separate features from labels
X = df_train.drop(columns=["label", "attack_cat"])
y_binary     = df_train["label"].values          # 0=normal, 1=attack
y_multiclass = df_train["attack_cat"].values      # 10 attack types

print("Feature shape:", X.shape)
print("Binary labels:", np.unique(y_binary))
print("Multi labels: ", np.unique(y_multiclass))


In [ ]:
# Encode categorical columns
cat_cols = X.select_dtypes(include="object").columns.tolist()
print("Categorical columns:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

print("After encoding dtypes:\n", X.dtypes.value_counts())


In [ ]:
# Scale numeric features to [-1, 1]
# MinMaxScaler(-1, 1) matches the Generator's Tanh output range exactly
# (StandardScaler is unbounded, which biases Tanh-based generators).
scaler = MinMaxScaler(feature_range=(-1, 1))
X_scaled = scaler.fit_transform(X)

print("Scaled shape:", X_scaled.shape)
print("Min:", X_scaled.min(axis=0)[:5].round(3))
print("Max:", X_scaled.max(axis=0)[:5].round(3))


In [ ]:
# Train/test split — this held-out test set is real data only,
# and stays untouched so we can measure genuine before/after performance later.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("\nTrain attack ratio:", y_train.mean().round(3))
print("Test  attack ratio:", y_test.mean().round(3))


In [ ]:
# Isolate minority classes (for reference / future targeted augmentation)
minority_mask = np.isin(df_train["attack_cat"],
                        ["Worms", "Shellcode", "Backdoor", "Analysis"])

X_minority = X_scaled[minority_mask]
y_minority = df_train["attack_cat"].values[minority_mask]

print("Minority class counts:")
for cls in ["Worms", "Shellcode", "Backdoor", "Analysis"]:
    count = (y_minority == cls).sum()
    print(f"  {cls:15s}: {count}")

print("\n✅ Preprocessing complete")


## 2. WGAN — synthetic attack traffic generator

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print("Seed:", SEED)

INPUT_DIM  = X_train.shape[1]
LATENT_DIM = 64
HIDDEN_DIM = 256
BATCH_SIZE = 256

print(f"Input dim: {INPUT_DIM}")


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM * 2),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM * 2, INPUT_DIM),
            nn.Tanh()          # matches the [-1, 1] MinMaxScaler range above
        )
    def forward(self, z):
        return self.net(z)


class Critic(nn.Module):   # "Critic" not "Discriminator" — WGAN convention
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM, HIDDEN_DIM * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(HIDDEN_DIM * 2, HIDDEN_DIM),
            nn.LeakyReLU(0.2),
            nn.Linear(HIDDEN_DIM, 1)
            # no Sigmoid — WGAN critic outputs a raw (unbounded) score
        )
    def forward(self, x):
        return self.net(x)


G = Generator().to(device)
C = Critic().to(device)

opt_G = torch.optim.RMSprop(G.parameters(), lr=5e-5)
opt_C = torch.optim.RMSprop(C.parameters(), lr=5e-5)

print("Generator params:", sum(p.numel() for p in G.parameters()))
print("Critic params:   ", sum(p.numel() for p in C.parameters()))


In [ ]:
# Train the WGAN only on real attack samples
X_attack = X_train[y_train == 1]
print(f"Training WGAN on {len(X_attack)} attack samples")

dataset = TensorDataset(torch.FloatTensor(X_attack))
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)


In [ ]:
CLIP     = 0.01   # weight clipping (WGAN key trick, keeps critic 1-Lipschitz)
N_CRITIC = 5       # train critic 5x per generator update
GAN_EPOCHS = 20

g_losses, d_losses = [], []

for epoch in range(GAN_EPOCHS):
    g_epoch, d_epoch = 0, 0

    for (real_batch,) in loader:
        real_batch = real_batch.to(device)
        bs = real_batch.size(0)

        # ── Train Critic N_CRITIC times ──
        for _ in range(N_CRITIC):
            z    = torch.randn(bs, LATENT_DIM).to(device)
            fake = G(z).detach()

            loss_C = -(torch.mean(C(real_batch)) - torch.mean(C(fake)))
            opt_C.zero_grad(); loss_C.backward(); opt_C.step()

            for p in C.parameters():
                p.data.clamp_(-CLIP, CLIP)
            d_epoch += loss_C.item()

        # ── Train Generator once ──
        z    = torch.randn(bs, LATENT_DIM).to(device)
        fake = G(z)
        loss_G = -torch.mean(C(fake))
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
        g_epoch += loss_G.item()

    g_losses.append(g_epoch / len(loader))
    d_losses.append(d_epoch / (len(loader) * N_CRITIC))
    print(f"Epoch [{epoch+1:2d}/{GAN_EPOCHS}] | G loss: {g_losses[-1]:.4f} | C loss: {d_losses[-1]:.4f}")

print("\n✅ WGAN training complete")


In [ ]:
# Generate synthetic attack samples from the trained generator
G.eval()
with torch.no_grad():
    z = torch.randn(5000, LATENT_DIM).to(device)
    gan_samples = G(z).cpu().numpy()

print("GAN samples shape:", gan_samples.shape)
print("Mean:", gan_samples.mean().round(4))
print("Std: ", gan_samples.std().round(4))
print("Min: ", gan_samples.min().round(4))
print("Max: ", gan_samples.max().round(4))


## 3. Diffusion model — a second synthetic-attack generator

In [ ]:
T           = 100   # timesteps (reduced for speed)
DIFF_EPOCHS = 20

betas     = torch.linspace(1e-4, 0.02, T).to(device)
alphas    = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t):
    """Add noise to data at timestep t."""
    ab    = alpha_bar[t].view(-1, 1)
    noise = torch.randn_like(x0)
    xt    = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise
    return xt, noise


class DiffusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.time_emb = nn.Embedding(T, 32)
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM + 32, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, INPUT_DIM)
        )
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        return self.net(torch.cat([x, t_emb], dim=-1))


diff_model = DiffusionNet().to(device)
opt_diff   = torch.optim.Adam(diff_model.parameters(), lr=1e-3)
scheduler  = torch.optim.lr_scheduler.StepLR(opt_diff, step_size=10, gamma=0.5)

print("Diffusion model params:", sum(p.numel() for p in diff_model.parameters()))


In [ ]:
diff_losses = []

for epoch in range(DIFF_EPOCHS):
    epoch_loss = 0

    for (x0_batch,) in loader:
        x0_batch = x0_batch.to(device)
        bs       = x0_batch.size(0)

        t          = torch.randint(0, T, (bs,)).to(device)
        xt, noise  = q_sample(x0_batch, t)
        pred_noise = diff_model(xt, t)

        loss = nn.MSELoss()(pred_noise, noise)
        opt_diff.zero_grad(); loss.backward(); opt_diff.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)
    diff_losses.append(avg_loss)
    scheduler.step()

    print(f"Epoch [{epoch+1:2d}/{DIFF_EPOCHS}] | Loss: {avg_loss:.4f} "
          f"| {'Good' if avg_loss < 1.0 else 'High'}")

print("\n✅ Diffusion training complete")


In [ ]:
@torch.no_grad()
def ddpm_sample(n_samples):
    diff_model.eval()
    x = torch.randn(n_samples, INPUT_DIM).to(device)

    for t_step in reversed(range(T)):
        t_tensor = torch.full((n_samples,), t_step, dtype=torch.long).to(device)
        pred     = diff_model(x, t_tensor)

        b  = betas[t_step]
        a  = alphas[t_step]
        ab = alpha_bar[t_step]

        x = (1 / torch.sqrt(a)) * (x - (b / torch.sqrt(1 - ab)) * pred)
        if t_step > 0:
            x += torch.sqrt(b) * torch.randn_like(x)

    return x.cpu().numpy()

diff_samples = ddpm_sample(5000)
print("✅ Diffusion samples generated:", diff_samples.shape)


## 4. Combine synthetic data with real training data

In [ ]:
combined_synthetic = np.vstack([gan_samples, diff_samples])

X_final = np.vstack([X_train, combined_synthetic])
y_final = np.hstack([y_train, np.ones(len(combined_synthetic))])

print("Real training data:  ", X_train.shape)
print("GAN samples:         ", gan_samples.shape)
print("Diffusion samples:   ", diff_samples.shape)
print("-" * 40)
print("X_final:              ", X_final.shape)
print("y_final:              ", y_final.shape)
print(f"Total attacks:         {y_final.sum():.0f}")
print(f"Attack ratio:          {y_final.mean()*100:.1f}%")


In [ ]:
# Quick sanity check: does synthetic data land in a similar range/scale as real attacks?
print("── Synthetic Data Stats ──")
print(f"Mean:  {combined_synthetic.mean():.4f}")
print(f"Std:   {combined_synthetic.std():.4f}")
print(f"Min:   {combined_synthetic.min():.4f}")
print(f"Max:   {combined_synthetic.max():.4f}")

print("\n── Real Attack Data Stats ──")
print(f"Mean:  {X_attack.mean():.4f}")
print(f"Std:   {X_attack.std():.4f}")
print(f"Min:   {X_attack.min():.4f}")
print(f"Max:   {X_attack.max():.4f}")


## 5. Defender model — trained on real + synthetic data

Evaluated on the untouched real test set, and compared against a defender
trained on real data only, so there's actual evidence for whether augmentation
helps detection rather than just diversifying the training distribution.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

class Defender(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)


def train_defender(X_tr, y_tr, epochs=10):
    model = Defender().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit  = nn.BCELoss()

    X_t = torch.FloatTensor(X_tr).to(device)
    y_t = torch.FloatTensor(y_tr).unsqueeze(1).to(device)
    ds  = torch.utils.data.TensorDataset(X_t, y_t)
    dl  = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=True)

    for epoch in range(epochs):
        epoch_loss = 0
        for xb, yb in dl:
            pred = model(xb)
            loss = crit(pred, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        print(f"  Epoch [{epoch+1:2d}/{epochs}] | Loss: {epoch_loss/len(dl):.4f}")
    return model


def evaluate(model, X_te, y_te):
    model.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(X_te).to(device)
        scores = model(X_t).cpu().numpy().ravel()
    preds = (scores >= 0.5).astype(int)
    return {
        "precision": precision_score(y_te, preds, zero_division=0),
        "recall":    recall_score(y_te, preds, zero_division=0),
        "f1":        f1_score(y_te, preds, zero_division=0),
        "auc":       roc_auc_score(y_te, scores),
    }


print("Training defender on REAL data only...")
defender_real = train_defender(X_train, y_train)

print("\nTraining defender on REAL + SYNTHETIC (GAN + Diffusion) data...")
defender_aug = train_defender(X_final, y_final)

metrics_real = evaluate(defender_real, X_test, y_test)
metrics_aug  = evaluate(defender_aug, X_test, y_test)

print("\n" + "=" * 50)
print("HELD-OUT TEST SET PERFORMANCE (real data only, never used in training)")
print("=" * 50)
print(f"{'Metric':<12}{'Real only':>12}{'Real+Synth':>14}")
for k in metrics_real:
    print(f"{k:<12}{metrics_real[k]:>12.4f}{metrics_aug[k]:>14.4f}")

# use the augmented defender for the evolution experiment below
defender = defender_aug


## 6. Adversarial attack evolution

WGAN generates an initial attack population, diffusion "mutates" it over
several generations, and each generation is scored against the defender.
`mutate_attack` clamps every step to `[-1, 1]` so evolved samples stay within
the same bounds real data occupies, instead of drifting off-manifold.


In [ ]:
def mutate_attack(population, mutation_steps=30):
    """Evolve attacks using diffusion-guided mutation, clamped to the real data range."""
    x = population.clone()
    step_size = 0.02
    for _ in range(mutation_steps):
        t_val = torch.randint(0, T, (x.size(0),)).to(device)
        with torch.no_grad():
            noise_pred = diff_model(x, t_val)
        x = torch.clamp(x + step_size * noise_pred + 0.01 * torch.randn_like(x), -1.0, 1.0)
    return x


def predict_attack_evolution(n_generations=5, population_size=100, mutation_steps=30):
    print("=" * 60)
    print("   ATTACK EVOLUTION VS. DEFENDER — EMPIRICAL TEST")
    print("=" * 60)
    print(f"   Generations    : {n_generations}")
    print(f"   Population     : {population_size}")
    print(f"   Mutation Steps : {mutation_steps}")
    print("=" * 60)

    G.eval()
    with torch.no_grad():
        noise      = torch.randn(population_size, LATENT_DIM).to(device)
        population = G(noise)

    print(f"\nWGAN generated {population_size} initial attack samples")

    all_generations = {"Gen_0": population.cpu().numpy()}
    results = []

    for gen in range(n_generations):
        print(f"\n{'-'*60}")
        print(f"  GENERATION {gen+1} / {n_generations}")
        print(f"{'-'*60}")

        mutated = mutate_attack(population, mutation_steps=mutation_steps)

        defender.eval()
        with torch.no_grad():
            scores = defender(mutated).squeeze()

        evaded       = scores < 0.5
        caught       = scores >= 0.5
        evasion_rate = evaded.float().mean().item()

        normalized = mutated / (mutated.norm(dim=1, keepdim=True) + 1e-8)
        diversity  = (1 - (normalized @ normalized.T).mean()).item()

        gen0     = torch.FloatTensor(all_generations["Gen_0"]).to(device)
        mut_dist = (mutated - gen0).abs().mean().item()

        if   evasion_rate >= 0.7: threat = "CRITICAL"
        elif evasion_rate >= 0.4: threat = "HIGH"
        elif evasion_rate >= 0.1: threat = "MEDIUM"
        else:                     threat = "LOW"

        print(f"  Attacks Evaded Defender  : {evaded.sum().item():.0f}  ({evasion_rate:.1%})")
        print(f"  Attacks Caught           : {caught.sum().item():.0f}  ({1-evasion_rate:.1%})")
        print(f"  Diversity Score          : {diversity:.4f}")
        print(f"  Mutation Distance        : {mut_dist:.4f}")
        print(f"  Threat Level             : {threat}")

        all_generations[f"Gen_{gen+1}"] = mutated.cpu().numpy()
        results.append({
            "Generation": gen + 1, "Evasion Rate": round(evasion_rate, 4),
            "Evaded": int(evaded.sum().item()), "Caught": int(caught.sum().item()),
            "Diversity": round(diversity, 4), "Mutation Dist": round(mut_dist, 4),
            "Threat Level": threat
        })

        # keep top evaders, refill with fresh GAN samples
        n_survivors = population_size // 2
        top_indices = (1 - scores).topk(n_survivors).indices
        survivors   = mutated[top_indices]

        G.eval()
        with torch.no_grad():
            noise   = torch.randn(population_size - n_survivors, LATENT_DIM).to(device)
            new_gen = G(noise)

        population = torch.cat([survivors, new_gen], dim=0)

    results_df = pd.DataFrame(results)
    print(f"\n{'='*60}")
    print("  FINAL SUMMARY")
    print(f"{'='*60}")
    print(results_df.to_string(index=False))

    return results_df, all_generations


results_df, all_generations = predict_attack_evolution()


## 6a. Per-feature realism diagnostic

Checks how many mutated features fall outside the real attack data's *actual*
observed per-feature min/max (a tighter bound than the global `[-1, 1]` clamp),
and tracks average feature drift per generation. Near-constant real features
(range < 1e-3) are masked out of the drift average instead of being divided
by, which previously caused the mean to blow up into the thousands.


In [ ]:
real_min = X_attack.min(axis=0)
real_max = X_attack.max(axis=0)
real_range = real_max - real_min

# Mask near-constant features (avoids division by ~0 dominating the mean)
real_range_safe = np.where(real_range < 1e-3, np.nan, real_range)

print(f"{'Generation':<12}{'% features out-of-range':>26}{'Mean feature drift':>22}")
for gen_name, gen_data in all_generations.items():
    below = gen_data < real_min
    above = gen_data > real_max
    out_of_range = (below | above)
    pct_out = out_of_range.mean() * 100

    drift = np.abs(gen_data - gen_data.mean(axis=0)) / real_range_safe
    mean_drift = np.nanmean(drift)

    print(f"{gen_name:<12}{pct_out:>25.2f}%{mean_drift:>22.4f}")


## 6b. Which features are the top evaders exploiting?

Turns "X% evasion" into a concrete, inspectable finding: which specific
features moved most for the samples that most confidently fooled the defender.


In [ ]:
final_gen_key = f"Gen_{len(results_df)}"
final_gen = all_generations[final_gen_key]
final_gen_t = torch.FloatTensor(final_gen).to(device)

defender.eval()
with torch.no_grad():
    final_scores = defender(final_gen_t).squeeze().cpu().numpy()

evader_idx = np.argsort(final_scores)[:5]   # 5 lowest scores = most confidently evaded
gen0 = all_generations["Gen_0"]

feature_names = X.columns.tolist()

print(f"Top 5 evading samples in {final_gen_key} — feature drift from Gen_0:\n")
for i in evader_idx:
    diffs = final_gen[i] - gen0[i]
    top_feats = np.argsort(-np.abs(diffs))[:5]  # 5 most-changed features
    print(f"Sample {i} | defender score: {final_scores[i]:.4f}")
    for f in top_feats:
        print(f"    {feature_names[f]:<20s} Gen0: {gen0[i][f]:+.3f}  ->  Final: {final_gen[i][f]:+.3f}  (Δ {diffs[f]:+.3f})")
    print()


## 7. Plots

In [ ]:
# Training loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(g_losses, label="Generator Loss", color="tomato")
axes[0].plot(d_losses, label="Critic Loss",    color="steelblue")
axes[0].set_title("WGAN Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(diff_losses, label="Diffusion Loss", color="green")
axes[1].set_title("Diffusion Training Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE Loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("GAN + Diffusion Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Class distribution before vs after augmentation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

_, counts_before = np.unique(y_train, return_counts=True)
axes[0].bar(["Normal", "Attack"], counts_before,
            color=["steelblue", "tomato"], edgecolor="black")
axes[0].set_title("Before Augmentation"); axes[0].set_ylabel("Count")
for i, c in enumerate(counts_before):
    axes[0].text(i, c + 100, str(c), ha="center", fontweight="bold")

_, counts_after = np.unique(y_final, return_counts=True)
axes[1].bar(["Normal", "Attack"], counts_after,
            color=["steelblue", "tomato"], edgecolor="black")
axes[1].set_title("After GAN + Diffusion Augmentation"); axes[1].set_ylabel("Count")
for i, c in enumerate(counts_after):
    axes[1].text(i, c + 100, str(c), ha="center", fontweight="bold")

plt.suptitle("Class Distribution Before vs After", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# t-SNE: real vs GAN vs diffusion samples
from sklearn.manifold import TSNE
import seaborn as sns

print("Running t-SNE (1-2 min)...")
n = 500
combined_viz = np.vstack([X_attack[:n], gan_samples[:n], diff_samples[:n]])
labels_viz   = ["Real"] * n + ["GAN"] * n + ["Diffusion"] * n

reduced = TSNE(n_components=2, random_state=42, init="pca").fit_transform(combined_viz)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=labels_viz,
                 palette={"Real": "steelblue", "GAN": "tomato", "Diffusion": "green"},
                 alpha=0.6, s=25)
plt.title("t-SNE: Real vs GAN vs Diffusion Attack Samples")
plt.tight_layout()
plt.show()


In [ ]:
# Evasion rate over generations — the key result
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(results_df["Generation"], results_df["Evasion Rate"],
             marker="o", color="tomato", linewidth=2)
axes[0].axhline(y=0.7, color="red",    linestyle="--", label="Critical (0.7)")
axes[0].axhline(y=0.4, color="orange", linestyle="--", label="High (0.4)")
axes[0].set_title("Evasion Rate per Generation")
axes[0].set_xlabel("Generation"); axes[0].set_ylabel("Evasion Rate")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(results_df["Generation"], results_df["Diversity"],
             marker="o", color="purple", linewidth=2)
axes[1].set_title("Population Diversity per Generation")
axes[1].set_xlabel("Generation"); axes[1].set_ylabel("Diversity Score")
axes[1].grid(True, alpha=0.3)

axes[2].plot(results_df["Generation"], results_df["Mutation Dist"],
             marker="o", color="steelblue", linewidth=2)
axes[2].set_title("Mutation Distance from Gen 0")
axes[2].set_xlabel("Generation"); axes[2].set_ylabel("Distance")
axes[2].grid(True, alpha=0.3)

plt.suptitle("Adversarial Evolution vs. Defender", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Aggregate: which features evaders exploit most often ──────────
# Look at ALL evading samples (not just top 5), and for each one find
# its top-5 most-drifted features. Then count how often each feature
# shows up across the whole evading population.

from collections import Counter

evaded_mask = final_scores < 0.5
evaded_indices = np.where(evaded_mask)[0]
print(f"Aggregating over {len(evaded_indices)} evading samples in {final_gen_key}\n")

feature_counter = Counter()
for i in evaded_indices:
    diffs = final_gen[i] - gen0[i]
    top_feats = np.argsort(-np.abs(diffs))[:5]
    for f in top_feats:
        feature_counter[feature_names[f]] += 1

top_n = 15
most_common = feature_counter.most_common(top_n)
names, counts = zip(*most_common)

plt.figure(figsize=(9, 6))
plt.barh(names[::-1], counts[::-1], color="tomato", edgecolor="black")
plt.xlabel(f"Times appearing in top-5 drifted features (out of {len(evaded_indices)} evaders)")
plt.title(f"Most Frequently Exploited Features — {final_gen_key} Evaders")
plt.tight_layout()
plt.show()

print("Top features by frequency:")
for name, count in most_common:
    pct = count / len(evaded_indices) * 100
    print(f"  {name:<20s} {count:>4d}  ({pct:.1f}% of evaders)")